# 01｜第 1 節：預測小考分數（線性迴歸）

**必做只有 4 組練習。** 每組先看上方 Markdown 的程式範例，再把它輸入到下方的練習 Cell，按 **Shift + Enter** 執行。學會完整流程後，才執行最後的 **Optional（選做）**。

資料流程：**讀取資料 → 選擇 X／y 並切分 → 補值＋標準化＋獨熱編碼 → 訓練、預測與評估**。

- 請先把 CSV 上傳到 Colab 的 `/content`；若在 VS Code 執行，將 CSV 放在 Notebook 的工作目錄。
- 先執行一次下方的「匯入套件」Cell；其餘必做程式由自己照範例輸入。
- Colab 一般沒有 `Microsoft JhengHei`，因此 Colab 圖表使用英文標籤避免中文字型亂碼。


In [ ]:
# Import the libraries used in this notebook.
# 匯入本份 Notebook 會用到的套件。
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Use the requested Chinese font on systems where it is installed.
# 在已安裝微軟正黑體的系統使用此字型。
matplotlib.rc('font', family='Microsoft JhengHei')

# Draw minus signs using a widely supported character.
# 修正圖表座標軸的負號顯示。
matplotlib.rcParams['axes.unicode_minus'] = False

# Colab normally does not include Microsoft JhengHei; use English plot labels there.
# Colab 通常沒有微軟正黑體，這裡改用內建字型並使用英文圖表標籤。
if Path('/content').exists():
    matplotlib.rc('font', family='DejaVu Sans')

## 資料集欄位

| 欄位 | 內容 |
|---|---|
| `student_id` | 學生編號，不作為模型輸入 |
| `study_hours` | 每日讀書時數（含缺漏與一筆錯誤的 60 小時） |
| `practice_questions` | 每日練習題數（含缺漏） |
| `learning_method` | 自習、讀書會、線上課 |
| `attendance_rate` | 出席率，保留供延伸使用 |
| `sleep_hours` | 睡眠時數，保留供延伸使用 |
| `effort_level` | 學習投入程度，保留供延伸使用 |
| `quiz_score` | 小考分數：第 1 節迴歸預測目標 |
| `passed` | 是否及格（0：未及格，1：及格）：第 2 節分類目標 |

**本節只使用** `study_hours`、`practice_questions`、`learning_method` 預測 `quiz_score`。


## 必做：4 組練習


### 練習 1／4｜讀取資料

開啟 CSV，看前 5 筆。已知 60 小時的紀錄不合理，先當成缺漏值。

**參考程式：**

```python
# Read the CSV file.
# 讀取 CSV 資料。
df = pd.read_csv('01_機器學習_學生練習資料_500筆.csv')

# Mark impossible daily study hours as missing.
# 把不合理的每日讀書時數當作缺漏值。
df.loc[df['study_hours'] > 24, 'study_hours'] = np.nan

# View the first five rows.
# 查看前五筆資料。
display(df.head())
```


In [ ]:
# 請在此處根據範例練習程式碼



### 練習 2／4｜選擇 X、y 並切分資料

兩個數值欄與一個類別欄是輸入 X；小考分數是答案 y。先切成訓練集與測試集。

**參考程式：**

```python
# Select the features and the target.
# 選擇輸入特徵與預測目標。
X = df[['study_hours', 'practice_questions', 'learning_method']]
y = df['quiz_score']

# Keep 20% of the rows for testing.
# 留下 20% 的資料作為測試集。
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Show the training and test sizes.
# 顯示訓練集與測試集筆數。
print('Train:', len(X_train), 'Test:', len(X_test))
```


In [ ]:
# 請在此處根據範例練習程式碼


### 練習 3／4｜補值、獨熱編碼、標準化（同一個練習）

在同一個 Cell 中，依序完成 **① 補缺漏值 → ② One-Hot 獨熱編碼 → ③ StandardScaler 標準化**。

中位數只從訓練集計算；先補好缺漏值，再做標準化，避免把多個步驟塞在同一行。測試集沿用訓練集的中位數與標準化規則。這份資料固定切分後，兩組都有三種學習方式，因此練習中可直接使用 `pd.get_dummies()`。

**參考程式：**

```python
# Select numerical features.
# 指定數值型特徵。
num_cols = ['study_hours', 'practice_questions']

# Calculate medians from training data.
# 使用訓練集計算中位數。
medians = X_train[num_cols].median()

# Fill missing values before scaling.
# 先補齊訓練集與測試集的缺漏值。
X_train_ready = X_train.fillna(medians)
X_test_ready = X_test.fillna(medians)

# Convert categories into 0/1 columns.
# 將類別資料轉成 0／1 欄位。
X_train_ready = pd.get_dummies(X_train_ready, dtype=int)
X_test_ready = pd.get_dummies(X_test_ready, dtype=int)

# Create a scaler.
# 建立標準化工具。
scaler = StandardScaler()

# Scale numerical columns using training data.
# 使用訓練集建立規則，再轉換數值欄位。
X_train_ready[num_cols] = scaler.fit_transform(X_train_ready[num_cols])
X_test_ready[num_cols] = scaler.transform(X_test_ready[num_cols])

# Preview the results.
# 檢視前五筆完成前處理的特徵。
display(X_train_ready.head())
```


In [ ]:
# 請在此處根據範例練習程式碼


### 練習 4／4｜模型訓練、預測與評估

使用 `LinearRegression`。**MAE／RMSE 越小越好；R² 越接近 1 通常越好**（也可能小於 0）。

**參考程式：**

```python
# Train a linear regression model.
# 訓練線性迴歸模型。
model = LinearRegression().fit(X_train_ready, y_train)

# Predict the unseen test scores.
# 預測測試資料的小考分數。
pred = model.predict(X_test_ready)

# Calculate regression metrics.
# 計算迴歸評估指標。
mse = mean_squared_error(y_test, pred)

print('MSE:', round(mse, 2))
print('MAE:', round(mean_absolute_error(y_test, pred), 2))
print('RMSE:', round(np.sqrt(mse), 2))
print('R²:', round(r2_score(y_test, pred), 2))
```


In [ ]:
# 請在此處根據範例練習程式碼


## 完成必做後的觀察

你已完成從 CSV 到模型的完整流程：`X_train_ready` 是處理後的 Pandas 特徵表格，`model.fit()` 代表訓練，`model.predict()` 產生預測，四個指標用來判斷預測成效。接下來都是 **Optional**，不需要手動照寫，直接執行即可。


### Optional A｜看預測散佈圖

藍點越接近虛線代表預測越接近實際值；虛線上方表示高估，下方表示低估。


In [ ]:
# Draw actual scores against predicted scores.
# 將實際分數與預測分數畫成散佈圖。
plt.scatter(y_test, pred, alpha=0.6)

# Draw the perfect-prediction reference line.
# 畫出預測值等於實際值的參考線。
plt.plot([0, 100], [0, 100], '--', color='orange')
plt.xlabel('Actual score')
plt.ylabel('Predicted score')
plt.title('Actual vs. Predicted')
plt.show()

### Optional B｜輸入三位新同學，預測分數

沿用已訓練的 `scaler` 與 `model`，**不重新訓練模型**。新資料不一定包含所有學習方式，因此這裡使用 `reindex()` 對齊訓練時的欄位；這是新資料預測時的保護步驟，不是每次做 One-Hot 都必須使用。

In [ ]:
# Create three new examples.
# 輸入三筆新的學習紀錄。
new_data = pd.DataFrame({
    'study_hours': [1.5, 3.0, 5.0],
    'practice_questions': [5, 12, 22],
    'learning_method': ['自習', '讀書會', '線上課']
})

# Fill missing values using training medians.
# 先使用訓練集的中位數補齊缺漏值。
X_new_ready = new_data.fillna(medians)

# Convert categories into 0/1 columns.
# 將類別型資料轉換成 0／1 欄位。
X_new_ready = pd.get_dummies(X_new_ready, dtype=int)

# Apply the original scaling rules.
# 使用原本的標準化規則轉換數值欄位。
X_new_ready[num_cols] = scaler.transform(X_new_ready[num_cols])

# Match the training feature columns.
# 對齊訓練模型時使用的特徵欄位。
X_new_ready = X_new_ready.reindex(columns=X_train_ready.columns, fill_value=0)

# Predict with the already trained model.
# 使用已訓練的模型預測三筆新資料。
new_data['predicted_score'] = model.predict(X_new_ready).round(1)

display(new_data)

### Optional C｜讀書時數和練習題數相關嗎？

Pearson 相關係數介於 -1 到 1。相關程度高，不表示某個特徵一定要刪除，也不等於因果關係。


In [ ]:
# Check the correlation between the two numerical features.
# 檢查兩個數值特徵的 Pearson 相關係數。
print(df[['study_hours', 'practice_questions']].corr().round(2))


**第 1 節完成。** 如果時間不足，完成 4 組必做即可；Optional 可留待課後。
